# What Fraction of Humans Have Ever...?

Monte Carlo estimation: sample random people from all of human history, then ask an LLM
whether each person would have experienced various things.

Uses the same 100K sample as the Most Famous Person analysis.

In [ ]:
import numpy as np
import dill
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from collections import defaultdict
import random

import sys
import os
sys.path.insert(0, '..')
os.chdir('..')

from person import sample_person
from llm_utils import get_client, make_langchain_messages, CostTracker, extract_json
from export import get_continent

os.chdir('WhimsicalQuestions')

from questions import SAMPLING_QUESTIONS

## 1. Load the 100K sample

In [ ]:
SAMPLE_FILE = '../FamousPerson/famous_person_sample_100000.pkl'

print(f'Loading sample from {SAMPLE_FILE}...')
with open(SAMPLE_FILE, 'rb') as f:
    people = dill.load(f)
print(f'Loaded {len(people):,} people')

# Quick look
for i, p in enumerate(people[:10]):
    if p.era == 'Paleolithic':
        print(f"{p.birth_year_str:>12s} \t {str(p.age_at_death):>5s} \t {p.sex} \t {p.region}")
    else:
        loc = p.location
        print(f"{p.birth_year_str:>12s} \t {str(p.age_at_death):>5s} \t {p.sex} \t {loc.subregion}, {loc.country}")

In [ ]:
# Compute easy answers
print(f"Easy questions (known from data)")
print(f"Total humans ever born: {TOTAL_HUMANS_EVER_BORN:,.0f}")
print(f"\n{'Question':<35s} {'Count':>15s} {'Fraction':>12s}")
print("-" * 65)

# Special case: survived past infancy — compute from sample
n_survived = sum(1 for p in people if p.age_at_death == 'alive' or p.age_at_death >= 1)
infancy_frac = n_survived / len(people)
EASY_QUESTIONS['survived_past_infancy']['known_count'] = int(infancy_frac * TOTAL_HUMANS_EVER_BORN)

for qid, q in EASY_QUESTIONS.items():
    count = q['known_count']
    frac = count / TOTAL_HUMANS_EVER_BORN
    print(f"{q['short']:<35s} {count:>15,} {frac:>11.4%}")

In [ ]:
def get_death_year(person):
    """Return the year this person died (or 2026 if alive)."""
    if person.age_at_death == 'alive':
        return 2026
    return person.birth_year + person.age_at_death


def get_era(person):
    by = person.birth_year
    if by < -10000:    return "pre-10000"
    elif by < -500:    return "-10000 to -500"
    elif by < 0:       return "-500 to 0"
    elif by < 600:     return "0-600"
    elif by < 1500:    return "600-1500"
    elif by < 1800:    return "1500-1800"
    elif by < 1900:    return "1800-1900"
    elif by < 1950:    return "1900-1950"
    elif by < 1980:    return "1950-1980"
    else:              return "1980+"


def get_fine_bin(person):
    """Fine bin: (era, country/region)."""
    country = person.location.country if person.era == 'Holocene' else person.region
    return (get_era(person), country)


def get_continent_for_person(person):
    country = person.location.country if person.era == 'Holocene' else None
    region = person.region if person.era == 'Paleolithic' else None
    return get_continent(country, region)


def get_applicable_questions(person):
    """Return list of questions that need LLM estimation for this person."""
    death_yr = get_death_year(person)
    applicable = []
    for q in SAMPLING_QUESTIONS:
        # Check earliest_year (known zero if person died before this)
        if q.earliest_year is not None and death_yr < q.earliest_year:
            continue
        # Check demographic filter
        if q.filter_fn is not None and not q.filter_fn(person):
            continue
        # Check known_answer_fn
        if q.known_answer_fn is not None and q.known_answer_fn(person) is not None:
            continue
        applicable.append(q)
    return applicable


# ── Show question applicability across the sample ──
print(f"Question applicability across {len(people):,} people:\n")
print(f"{'Question':<40s} {'Applicable':>10s} {'% of pop':>8s}")
print("-" * 60)

# Count how many people each question applies to
for q in SAMPLING_QUESTIONS:
    n_applicable = 0
    for p in people:
        death_yr = get_death_year(p)
        if q.earliest_year is not None and death_yr < q.earliest_year:
            continue
        if q.filter_fn is not None and not q.filter_fn(p):
            continue
        n_applicable += 1
    print(f"{q.short:<40s} {n_applicable:>10,} {n_applicable/len(people):>7.1%}")

In [ ]:
# ── Fine bins (era × country/region) ──
fine_bins = defaultdict(list)
for i, p in enumerate(people):
    fine_bins[get_fine_bin(p)].append(i)
print(f"Fine bins: {len(fine_bins)}")

# How many questions does each person get asked?
questions_per_person = [len(get_applicable_questions(p)) for p in people]
n_queryable = sum(1 for n in questions_per_person if n > 0)
print(f"People with ≥1 applicable question: {n_queryable:,} ({n_queryable/len(people):.1%})")
print(f"Questions per person: min={min(questions_per_person)}, "
      f"median={int(np.median(questions_per_person))}, max={max(questions_per_person)}")

## 4. Neyman allocation and sampling

In [ ]:
# ── Neyman allocation ──
# Since we don't have prior variance estimates, use uniform allocation
# weighted by bin population (equivalent to proportional allocation).
# After the first run, we can switch to Neyman with estimated variances.

BUDGET = 800  # total LLM calls
MIN_DRAWS = 3

# Start with uniform sigma (proportional allocation)
UNIFORM_SIGMA = 0.15  # placeholder — all bins get same sigma

fine_keys = list(fine_bins.keys())
fine_sizes = np.array([len(fine_bins[k]) for k in fine_keys])

# ── Three-level merge (same as FamousPerson) ──
neyman_raw = fine_sizes * UNIFORM_SIGMA  # with uniform sigma, this is just proportional
n_alloc_init = BUDGET * neyman_raw / neyman_raw.sum()

# Level 1: country bins with enough allocation stay
final_bins = {}
continent_overflow = defaultdict(list)

for i, key in enumerate(fine_keys):
    if n_alloc_init[i] >= MIN_DRAWS:
        final_bins[key] = fine_bins[key]
    else:
        era = key[0]
        sample_p = people[fine_bins[key][0]]
        continent = get_continent_for_person(sample_p)
        continent_overflow[(era, continent)].extend(fine_bins[key])

# Level 2: continent bins
era_overflow = defaultdict(list)
for cont_key, members in continent_overflow.items():
    estimated_alloc = BUDGET * len(members) * UNIFORM_SIGMA / neyman_raw.sum()
    if estimated_alloc >= MIN_DRAWS and len(members) >= MIN_DRAWS:
        final_bins[cont_key] = members
    else:
        era_overflow[cont_key[0]].extend(members)

# Level 3: era-merged bins
for era, members in era_overflow.items():
    if members:
        final_bins[(era, '_merged')] = members

# ── Final allocation ──
final_keys = list(final_bins.keys())
final_sizes = np.array([len(final_bins[k]) for k in final_keys])

# Proportional allocation (uniform sigma)
neyman_final = final_sizes * UNIFORM_SIGMA
n_alloc_final = BUDGET * neyman_final / neyman_final.sum()
final_n_draws = np.array([max(MIN_DRAWS, round(n)) for n in n_alloc_final])

# Trim if over budget
while final_n_draws.sum() > BUDGET * 1.1:
    idx = np.argmax(final_n_draws)
    if final_n_draws[idx] > MIN_DRAWS:
        final_n_draws[idx] -= 1
    else:
        break

# Sample people from each bin
random.seed(42)
to_query = []

for b, key in enumerate(final_keys):
    members = final_bins[key]
    n_draws = min(final_n_draws[b], len(members))
    final_n_draws[b] = n_draws
    chosen = random.sample(members, n_draws)
    for idx in chosen:
        # Only query if person has at least 1 applicable question
        if questions_per_person[idx] > 0:
            to_query.append((people[idx], b, idx))

print(f"Final bins: {len(final_bins)} (from {len(fine_bins)} fine bins)")
print(f"Budget: {BUDGET}, actual LLM calls: {len(to_query)}")
print(f"Draws per bin: min={final_n_draws.min()}, "
      f"median={int(np.median(final_n_draws))}, max={final_n_draws.max()}")

# Show bins
print(f"\n{'Bin':<45s} {'Pop':>6s} {'Draws':>6s}")
print("-" * 60)
for b, key in enumerate(final_keys):
    label = f"{key[0]}, {key[1]}"
    print(f"{label:<45s} {final_sizes[b]:>6d} {final_n_draws[b]:>6d}")
print(f"\n{'TOTAL':<45s} {final_sizes.sum():>6d} {final_n_draws.sum():>6d}")

## 5. LLM query

In [ ]:
def format_person_light(person):
    """Lightweight person summary for the prompt."""
    lines = [
        f"Era: {person.era}",
        f"Birth year: {person.birth_year_str}",
        f"Age at death: {person.age_at_death}",
        f"Sex: {person.sex}"
    ]
    if person.era == 'Paleolithic':
        lines.append(f"Region: {person.region}")
    else:
        loc = person.location
        lines.append(f"Country: {loc.country}")
        lines.append(f"Subregion: {loc.subregion}")
    return "\n".join(lines)


def build_prompt(person):
    """Build prompt asking about all applicable questions for this person.
    
    Returns (messages, applicable_questions) where applicable_questions is the
    filtered list of Question objects used in the prompt.
    """
    applicable = get_applicable_questions(person)
    question_list = "\n".join(
        f"{i+1}. {q.prompt_text}" for i, q in enumerate(applicable)
    )
    
    messages = [
        {"role": "user", "content": f"""Below is a randomly sampled person from human history. For each question, estimate the probability that this person would answer "yes" (or that the statement is true of them).

Consider carefully:
- The person's birth year, location, sex, and age at death
- What was realistically available and common for someone of their specific time, place, and social position
- Most people throughout history were subsistence farmers with limited access to goods, travel, and information
- Even in the modern era, access varies enormously by country, rural/urban setting, and wealth
- Don't assume modern Western norms — think about what life was actually like for this specific person

Person:
{format_person_light(person)}

Questions:
{question_list}

First give brief reasoning for each question, then output a JSON object mapping each question number to the probability (0 to 1):
{{"1": 0.15, "2": 0.05, ...}}"""}
    ]
    return messages, applicable


# Preview a prompt
if to_query:
    # Find a modern person for a good preview
    preview_person = None
    for person, b, idx in to_query:
        if person.birth_year > 1900:
            preview_person = person
            break
    if preview_person is None:
        preview_person = to_query[0][0]
    messages, applicable = build_prompt(preview_person)
    print(f"Questions for this person: {len(applicable)} of {len(SAMPLING_QUESTIONS)}\n")
    print(messages[0]['content'])

In [ ]:
MODEL = 'gpt-5.2'
WORKERS = 30

# results[i] = (person, probs_dict, raw_text, bin_idx, person_idx)
# probs_dict maps question_id -> probability
results = [None] * len(to_query)


def query_one(i, person, bin_idx, person_idx):
    """Query LLM for one person."""
    client = get_client(MODEL)
    tracker = CostTracker(MODEL)
    
    messages, applicable = build_prompt(person)
    lc_messages = make_langchain_messages(messages)
    response = client.invoke(lc_messages, config={"callbacks": [tracker]})
    raw_text = response.content.strip()
    probs = extract_json(raw_text)
    
    # Map from prompt numbering to question IDs
    probs_by_id = {}
    for j, q in enumerate(applicable):
        key = str(j + 1)
        if key in probs:
            probs_by_id[q.id] = probs[key]
    
    results[i] = (person, probs_by_id, raw_text, bin_idx, person_idx)


with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(query_one, i, person, bin_idx, person_idx): i
        for i, (person, bin_idx, person_idx) in enumerate(to_query)
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc='Querying LLM'):
        future.result()

print(f"\nCompleted {len(results)} LLM calls across {len(final_bins)} bins")

In [ ]:
# Save raw results
RESULTS_FILE = 'whimsical_results.pkl'
with open(RESULTS_FILE, 'wb') as f:
    dill.dump(results, f)
print(f'Saved {len(results)} results to {RESULTS_FILE}')

## 6. Stratified estimation

For each question independently:
1. Compute per-bin means and variances from draws that included that question
2. Weight by bin population (accounting for known-zero populations)
3. Compute SE and confidence intervals

In [ ]:
n_total = len(people)
n_questions = len(SAMPLING_QUESTIONS)
n_bins = len(final_keys)

# ── For each question, compute the "known zero" population ──
# These are people for whom the answer is definitionally 0
# (before earliest_year or failing filter_fn)

question_ids = [q.id for q in SAMPLING_QUESTIONS]
q_idx = {q.id: i for i, q in enumerate(SAMPLING_QUESTIONS)}

# For each question, what fraction of the total population is queryable?
# (Not known-zero, passes filter)
queryable_count = np.zeros(n_questions)
for p in people:
    death_yr = get_death_year(p)
    for qi, q in enumerate(SAMPLING_QUESTIONS):
        if q.earliest_year is not None and death_yr < q.earliest_year:
            continue
        if q.filter_fn is not None and not q.filter_fn(p):
            continue
        queryable_count[qi] += 1

print(f"Queryable population per question:")
for qi, q in enumerate(SAMPLING_QUESTIONS):
    print(f"  {q.short:<40s} {queryable_count[qi]:>7,.0f} ({queryable_count[qi]/n_total:>5.1%})")

In [ ]:
# ── Group results by bin, per question ──
# For each bin and question, collect the probabilities from draws where that
# question was asked.

# bin_draws[bin_idx][question_id] = list of probabilities
bin_draws = defaultdict(lambda: defaultdict(list))

for person, probs_by_id, raw_text, bin_idx, person_idx in results:
    for qid, prob in probs_by_id.items():
        bin_draws[bin_idx][qid].append(prob)

# ── Per-bin, per-question: mean, variance, effective n ──
bin_means = np.zeros((n_bins, n_questions))
bin_vars = np.zeros((n_bins, n_questions))
bin_n_effective = np.zeros((n_bins, n_questions), dtype=int)

for b in range(n_bins):
    for qi, q in enumerate(SAMPLING_QUESTIONS):
        obs = bin_draws[b].get(q.id, [])
        n_h = len(obs)
        bin_n_effective[b, qi] = n_h
        if n_h >= 1:
            bin_means[b, qi] = np.mean(obs)
        if n_h >= 2:
            bin_vars[b, qi] = np.var(obs, ddof=1)

# ── Stratified estimation ──
# For each question:
#   Within the queryable population, the stratified estimator is:
#     ȳ_q = (1/N_queryable) × Σ_h N_h_queryable × ȳ_h
#   But we don't know N_h_queryable exactly (we know it at the full sample level).
#   
#   Simpler approach: treat the bin population as a proxy.
#   The fraction of bin h that is queryable for question q is approximately
#   the same as the fraction of draws in bin h that got question q.
#   
#   We compute:
#     θ̂_q = (1/N) × Σ_h N_h × (n_h_q / n_h) × ȳ_h_q
#   where n_h_q is the number of draws in bin h that got question q,
#   and n_h is the total draws in bin h.
#   
#   Actually, the cleanest approach: treat draws where a question wasn't asked
#   as known zeros for that question. This is correct because the person was
#   excluded precisely because the answer is 0 for them.

# Count total draws per bin
bin_total_draws = np.zeros(n_bins, dtype=int)
for person, probs_by_id, raw_text, bin_idx, person_idx in results:
    bin_total_draws[bin_idx] += 1

# For questions where a draw didn't include the question, the implicit value is 0.
# So the effective mean including implicit zeros is:
#   mean_including_zeros = (n_h_q × ȳ_h_q) / n_h_total
# And the effective variance (for SE calculation) accounts for the mix.

W_h = final_sizes  # bin population sizes

estimates = np.zeros(n_questions)
standard_errors = np.zeros(n_questions)

for qi, q in enumerate(SAMPLING_QUESTIONS):
    # Point estimate: weighted average of bin means (including implicit zeros)
    weighted_sum = 0.0
    var_sum = 0.0
    
    for b in range(n_bins):
        n_h = bin_total_draws[b]
        if n_h == 0:
            continue
        
        n_h_q = bin_n_effective[b, qi]  # draws that got this question
        
        # The bin mean INCLUDING implicit zeros for non-queried draws
        if n_h_q > 0:
            sum_probs = bin_means[b, qi] * n_h_q
        else:
            sum_probs = 0.0
        
        # Effective mean for all n_h draws (some queried, some implicitly 0)
        mean_all = sum_probs / n_h
        weighted_sum += W_h[b] * mean_all
        
        # Variance: mix of queried values and zeros
        # If we have values x_1,...,x_k (queried) and k zeros (non-queried),
        # the sample variance of all n_h values is:
        #   s² = [Σ(x_i - mean_all)² + (n_h - n_h_q) × mean_all²] / (n_h - 1)
        if n_h >= 2:
            if n_h_q > 0:
                obs = bin_draws[b].get(q.id, [])
                ss_queried = sum((x - mean_all)**2 for x in obs)
                ss_zeros = (n_h - n_h_q) * mean_all**2
                s2 = (ss_queried + ss_zeros) / (n_h - 1)
            else:
                s2 = 0.0
            var_sum += W_h[b]**2 * s2 / n_h
    
    estimates[qi] = weighted_sum / n_total
    standard_errors[qi] = np.sqrt(var_sum) / n_total

# ── Display results ──
ranking = sorted(range(n_questions), key=lambda i: estimates[i], reverse=True)

print(f"N={n_total:,}, bins={n_bins}, LLM calls={len(results)}")
print(f"\nEstimated fraction of all humans ever born who have...\n")
print(f"{'Rank':<5} {'Question':<40s} {'Frac':>8s} {'±1 SE':>8s} {'Queryable':>10s}")
print("-" * 75)
for rank, qi in enumerate(ranking, 1):
    q = SAMPLING_QUESTIONS[qi]
    print(f"{rank:<5} {q.short:<40s} {estimates[qi]:>7.2%} {standard_errors[qi]:>7.3%} "
          f"{queryable_count[qi]/n_total:>9.1%}")

## 7. Inspect individual responses

In [ ]:
# Show a few examples with full reasoning and labeled probabilities
for person, probs_by_id, raw_text, bin_idx, person_idx in results[:20]:
    print('=' * 70)
    key = final_keys[bin_idx]
    print(f"{format_person_light(person)}  [bin={key}]")
    print('-' * 70)
    # Print reasoning (everything before the JSON)
    json_start = raw_text.find('{')
    if json_start > 0:
        reasoning = raw_text[:json_start].strip()
        # Truncate very long reasoning
        if len(reasoning) > 500:
            reasoning = reasoning[:500] + '...'
        print(reasoning)
    print()
    # Print probabilities with question names
    for qid, prob in sorted(probs_by_id.items(), key=lambda x: x[1], reverse=True):
        qi = q_idx[qid]
        print(f"  {SAMPLING_QUESTIONS[qi].short:<40s} {prob:.0%}")
    print()

## 8. Save reweight data

In [ ]:
# Save all variables needed for reweighting / sensitivity analysis
reweight_data = {
    # Per-draw data
    "draws": [
        {
            "bin_idx": bin_idx,
            "probs": dict(probs_by_id),
            "applicable_questions": list(probs_by_id.keys()),
        }
        for person, probs_by_id, raw_text, bin_idx, person_idx in results
    ],
    
    # Bin structure
    "bins": {
        b: {"key": final_keys[b], "pop": int(final_sizes[b])}
        for b in range(len(final_keys))
    },
    
    # Population counts
    "n_total": n_total,
    
    # Question metadata
    "questions": [
        {"id": q.id, "short": q.short, "earliest_year": q.earliest_year}
        for q in SAMPLING_QUESTIONS
    ],
    
    # Per-person data for reweighting
    "birth_years": [p.birth_year for p in people],
    "death_years": [get_death_year(p) for p in people],
}

REWEIGHT_FILE = 'whimsical_reweight.pkl'
with open(REWEIGHT_FILE, 'wb') as f:
    dill.dump(reweight_data, f)
print(f'Saved reweight data to {REWEIGHT_FILE}')

## 9. Variance contributions by bin

In [ ]:
# Which bins contribute most to total variance?
# Average across questions

avg_var_per_bin = np.zeros(n_bins)
for b in range(n_bins):
    n_h = bin_total_draws[b]
    if n_h >= 2:
        total_var_contrib = 0.0
        n_counted = 0
        for qi, q in enumerate(SAMPLING_QUESTIONS):
            n_h_q = bin_n_effective[b, qi]
            if n_h_q > 0:
                obs = bin_draws[b].get(q.id, [])
                mean_all = sum(obs) / n_h
                ss_queried = sum((x - mean_all)**2 for x in obs)
                ss_zeros = (n_h - n_h_q) * mean_all**2
                s2 = (ss_queried + ss_zeros) / (n_h - 1)
                total_var_contrib += W_h[b]**2 * s2 / n_h
                n_counted += 1
        if n_counted > 0:
            avg_var_per_bin[b] = total_var_contrib / n_counted

total_avg_var = avg_var_per_bin.sum()

print(f"{'Bin':<45s} {'Pop':>6s} {'Draws':>6s} {'% of variance':>14s}")
print("-" * 75)

sorted_bins = sorted(range(n_bins), key=lambda b: avg_var_per_bin[b], reverse=True)
for b in sorted_bins[:30]:  # Top 30 bins
    key = final_keys[b]
    label = f"{key[0]}, {key[1]}"
    pct = 100 * avg_var_per_bin[b] / total_avg_var if total_avg_var > 0 else 0
    print(f"{label:<45s} {W_h[b]:>6d} {bin_total_draws[b]:>6d} {pct:>13.1f}%")

print(f"\n{'TOTAL':<45s} {W_h.sum():>6d} {bin_total_draws.sum():>6d} {'100.0%':>14s}")